# A1 — Groups: axioms, S₃, subgroups, conjugacy classes

**What this notebook claims** (state these without looking when the phase closes):

1. A group is a set with an associative binary operation, identity, and inverses; identity and inverses are unique.
2. S₃ (permutations of three objects) is the smallest non-abelian group: 6 elements, subgroups of orders 1, 2, 3, 6, conjugacy classes {e}, {3 transpositions}, {2 three-cycles}.
3. Lagrange: |H| divides |G|. A₃ (the 3-cycles + e) is normal; the order-2 subgroups are not; S₃/A₃ ≅ ℤ₂.
4. Conjugacy classes of Sₙ are exactly the cycle types.

Source map: Woit ch. 1, Zee part I. *Verify section numbers against your copy.*

## Definitions (group axioms)

A **group** $(G, \cdot)$ is a set $G$ with a binary operation satisfying:
(i) **closure**: $ab \in G$ for all $a,b$; (ii) **associativity**: $(ab)c = a(bc)$;
(iii) **identity**: $\exists e$ with $ea = ae = a$; (iv) **inverses**: $\forall a, \exists a^{-1}$ with $aa^{-1} = a^{-1}a = e$.

We verify each axiom *by construction* below for S₃ — nothing is assumed.

In [ ]:
from itertools import permutations, combinations

# --- permutations of (0,1,2) as tuples: p[i] = image of i ---
def compose(p, q):
    # (p*q)(i) = p(q(i)) — do q first, then p.
    return tuple(p[q[i]] for i in range(3))

def inverse(p):
    inv = [0]*3
    for i, j in enumerate(p):
        inv[j] = i
    return tuple(inv)

S3 = list(permutations(range(3)))
e = (0, 1, 2)

# Claim 1, brute-force check of all four axioms on the full set
assert len(S3) == 6
assert all(compose(p, q) in S3 for p in S3 for q in S3), "closure"
assert all(compose(compose(p, q), r) == compose(p, compose(q, r)) for p in S3 for q in S3 for r in S3), "associativity"
assert all(compose(e, p) == p and compose(p, e) == p for p in S3), "identity"
assert all(compose(p, inverse(p)) == e and compose(inverse(p), p) == e for p in S3), "inverses"
print("S3 satisfies all group axioms; |S3| =", len(S3))

# uniqueness of identity and inverse, as theorems require
identities = [p for p in S3 if all(compose(p, q) == q and compose(q, p) == q for q in S3)]
assert identities == [e], "identity is unique"
print("identity unique:", identities == [e])

In [ ]:
# Claim 2: the full Cayley table — build it from the operation, display it
labels = {p: "".join(str(i) for i in p) for p in S3}
print("Cayley table of S3 (rows = left factor):")
header = "     " + "  ".join(f"{labels[p]:>3}" for p in S3)
print(header)
for p in S3:
    row = "  ".join(f"{labels[compose(p, q)]:>3}" for q in S3)
    print(f"{labels[p]:>3} |{row}")

# non-abelian: find a commuting pair
noncomm = [(p, q) for p in S3 for q in S3 if compose(p, q) != compose(q, p)]
print(f"non-commuting pairs: {len(noncomm)}/36 -> S3 is non-abelian (smallest such group)")

## Claim 3: subgroups and Lagrange

**Lagrange's theorem:** if $H \le G$ then $|H|$ divides $|G|$ — the cosets partition $G$ into equal-size disjoint pieces.
We *find* every subgroup by brute force (all subsets whose size divides 6, closed under the operation), then check Lagrange holds and count them.

In [ ]:
# find all subgroups: subsets closed under * and inverses, of size dividing |G|
def is_subgroup(sub):
    sub = set(sub)
    return (e in sub
            and all(compose(p, q) in sub for p in sub for q in sub)
            and all(inverse(p) in sub for p in sub))

subgroups = []
for size in (1, 2, 3, 6):                      # only divisors of 6, by Lagrange
    for sub in combinations(S3, size):
        if is_subgroup(sub):
            subgroups.append(sub)
print(f"subgroups found: {len(subgroups)}  sizes: {sorted(len(s) for s in subgroups)}")
for s in sorted(subgroups, key=len):
    print(" ", sorted(labels[p] for p in s))
assert all(len(S3) % len(s) == 0 for s in subgroups), "Lagrange violated!" 

## Claim 4: conjugacy classes = cycle types; normality

Conjugation $g \mapsto hgh^{-1}$ partitions $G$ into **conjugacy classes**; in $S_n$ these are exactly the permutations of one cycle type (conjugation relabels points).
$H \trianglelefteq G$ (**normal**) iff it is a union of conjugacy classes iff its left and right cosets coincide.

In [ ]:
def conjugate(p, h):
    return compose(compose(h, p), inverse(h))

# build classes: orbit of each element under conjugation
seen, classes = set(), []
for g in S3:
    if g in seen:
        continue
    cls = frozenset(conjugate(g, h) for h in S3)
    seen |= cls
    classes.append(cls)
classes = sorted(classes, key=len)
print("conjugacy classes:")
for c in classes:
    print(f"  size {len(c)}: {sorted(labels[p] for p in c)}")

# cycle-type sanity check: transpositions are the permutations with exactly one fixed point
def n_fixed(p):
    return sum(1 for i in range(3) if p[i] == i)
transpositions = frozenset(p for p in S3 if n_fixed(p) == 1)
assert transpositions in classes, "classes do not match cycle types!"
print("classes == cycle types: OK")

# normality: a subgroup is normal iff it is a union of classes
def is_normal(sub):
    return all(conjugate(p, h) in sub for p in sub for h in S3)
for s in subgroups:
    print(f"H={sorted(labels[p] for p in s)}: normal = {is_normal(s)}")

In [ ]:
# S3 / A3 ~ Z2 : the quotient has two cosets with the group table of Z2
A3 = next(s for s in subgroups if len(s) == 3)
coset_H = frozenset(A3)
coset_other = frozenset(S3) - coset_H
def coset_mult(X, Y):
    return frozenset(compose(p, q) for p in X for q in Y)
assert coset_mult(coset_H, coset_H) == coset_H
assert coset_mult(coset_H, coset_other) == coset_other
assert coset_mult(coset_other, coset_H) == coset_other
assert coset_mult(coset_other, coset_other) == coset_H
print("quotient table matches Z2: OK  ->  S3/A3 ≅ Z2")

## My summary

*(Fill in after reading Woit ch. 1 / Zee part I. What is the one idea you would use to explain to a physics friend why group theory enters quantum mechanics?)*

-


## Gaps

*(Things you could not derive or state cold — convert each into a problem in problems/phaseA.md.)*

- 